In [ ]:
# %% [markdown]
# # Roboflow Flattened Video Reconstruction & YOLO Synchronization Tool

# %% [cell] Setup and Dependencies
import os
import shutil
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm

# 1. Define your directories
# Point this to your extracted Roboflow dataset folder containing the mixed images and .txt files
source_dir = Path("./raw_roboflow_dataset") 
output_dir = Path("./reconstructed_videos")
output_dir.mkdir(parents=True, exist_ok=True)

# Gather all images
image_extensions = (".jpg", ".jpeg", ".png")
all_images = sorted([p for p in source_dir.glob("*") if p.suffix.lower() in image_extensions])

print(f"Total images found: {len(all_images)}")

# %% [cell] Distance Metric Function
def compute_frame_distance(img_path_1, img_path_2):
    """
    Computes Mean Squared Error (MSE) between two frames after downscaling and 
    blurring. This dampens local motion (like walking) and focuses on global 
    structural and background consistency.
    """
    img1 = cv2.imread(str(img_path_1))
    img2 = cv2.imread(str(img_path_2))
    
    if img1 is None or img2 is None:
        return float('inf')
    
    # Resize to a small resolution for speed and motion invariance
    dim = (64, 64)
    img1 = cv2.resize(img1, dim)
    img2 = cv2.resize(img2, dim)
    
    # Convert to grayscale and blur slightly to account for walking/pixel shifts
    gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)
    gray1 = cv2.GaussianBlur(gray1, (5, 5), 0)
    gray2 = cv2.GaussianBlur(gray2, (5, 5), 0)
    
    mse = np.mean((gray1.astype("float") - gray2.astype("float")) ** 2)
    return mse

# %% [cell] Initialize Video Streams
# Set this to the number of distinct source videos mixed into your dataset folder
NUM_VIDEOS = 5 

remaining_images = set(all_images)
streams = []

# Seed the streams with the first N images
for i in range(NUM_VIDEOS):
    if remaining_images:
        seed_img = remaining_images.pop()
        streams.append([seed_img])

print(f"Initialized {len(streams)} video streams.")

# %% [cell] Iterative Greedy Reconstruction Loop
print("Reconstructing video sequences...")

with tqdm(total=len(all_images) - len(streams)) as pbar:
    while remaining_images:
        best_match_overall = None
        best_stream_idx = None
        min_distance = float('inf')
        
        # Compare remaining images against the tail (last frame) of active streams
        for stream_idx, stream in enumerate(streams):
            last_frame = stream[-1]
            
            for candidate in remaining_images:
                dist = compute_frame_distance(last_frame, candidate)
                if dist < min_distance:
                    min_distance = dist
                    best_match_overall = candidate
                    best_stream_idx = stream_idx
        
        # Append best match to its stream
        if best_match_overall is not None:
            streams[best_stream_idx].append(best_match_overall)
            remaining_images.remove(best_match_overall)
            pbar.update(1)
        else:
            break

print("Reconstruction complete!")

# %% [cell] Export, Rename, and Synchronize Labels
for stream_idx, stream in enumerate(streams):
    video_folder = output_dir / f"video_stream_{stream_idx + 1}"
    video_folder.mkdir(parents=True, exist_ok=True)
    
    for frame_idx, img_path in enumerate(stream):
        # Define clean sequential base names (e.g., frame_000000)
        new_base_name = f"frame_{frame_idx:06d}"
        
        # 1. Copy and rename the image file
        dest_img_path = video_folder / f"{new_base_name}{img_path.suffix}"
        shutil.copy(img_path, dest_img_path)
        
        # 2. Find and synchronize the matching YOLO .txt label file
        old_txt_path = img_path.with_suffix('.txt')
        if old_txt_path.exists():
            dest_txt_path = video_folder / f"{new_base_name}.txt"
            shutil.copy(old_txt_path, dest_txt_path)

print(f"All streams, sequential frames, and YOLO labels successfully exported to: {output_dir.absolute()}")